# sim-app low-level simulator tutorial

This notebook demonstrates the **low-level** `sim_app` API:
`simulate_3d_molecule_sphere_multires`, `simulate_3d_molecule_sphere_base`,
and `section_3d_molecule_sphere`.

Unlike `sim_app.generate_data(...)` (see `higher_level_api_tutorial.ipynb`),
which only exposes a curated subset of parameters and always builds exactly
one modality/axis pair, these functions take **every** simulation parameter
directly as a keyword argument. The whole point of this notebook: build one
plain Python dictionary with every parameter in it, and pass it straight
through with `**config` -- so you can see, and change, literally anything.

See [`../LOW_LEVEL_SIMULATOR.md`](../LOW_LEVEL_SIMULATOR.md) for the full
written parameter reference and the two-step vs. one-shot explanation this
notebook walks through.


## Setup

From the repository root, create the tutorial environment before opening this notebook:

```bash
bash env/create_tutorial_env.sh
conda activate sim-app-tutorial
jupyter-notebook --no-browser --ip=0.0.0.0 --port 8888
```


In [ ]:
from pathlib import Path
import sys

def find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "sim_app" / "__init__.py").is_file():
            return candidate
        nested = candidate / "sim_app"
        if (nested / "pyproject.toml").is_file() and (nested / "sim_app" / "__init__.py").is_file():
            return nested
    raise RuntimeError("Could not find the sim_app repository root.")

repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

sys.modules.pop("sim_app", None)
import sim_app
import numpy as np

print("Python:", sys.executable)
print("sim_app module:", getattr(sim_app, "__file__", "<no __file__>"))
print("has simulate_3d_molecule_sphere_multires:", hasattr(sim_app, "simulate_3d_molecule_sphere_multires"))


## The full parameter dictionary

Every keyword argument `simulate_3d_molecule_sphere_multires` accepts, laid
out as one dictionary -- nothing hidden, nothing implicit. Edit any value
below and re-run the notebook from this cell down.

The values here are deliberately small/fast for an interactive tutorial --
they are **not** the function's own defaults, and not the manuscript-scale
values used to generate the paper's benchmark data. See
[`../LOW_LEVEL_SIMULATOR.md`](../LOW_LEVEL_SIMULATOR.md) for both of those.

Parameters are grouped and ordered to match the manuscript's Methods section,
paragraph by paragraph -- each `# ====` header below names the paragraph
those parameters come from, in the same order the paragraphs appear.


In [ ]:
config = {
    # ==== 3D tissue geometry and cell placement ====
    "sphere_R_um": 200.0,               # tissue sphere radius, microns (R)
    "center": (0.0, 0.0, 0.0),          # tissue center, microns (c_0)
    "n_cells": 5000,                    # number of cells placed in the sphere
    "allow_cell_overlap": False,        # whether overlapping cell spheres are permitted during placement
    "cell_radius_kwargs": None,         # cell radius r_i ~ truncated log-normal; None -> lognormal(r_mean=6.0, r_sigma=0.35, r_min=3.0, r_max=12.0)

    # ==== Spatial domains with irregular boundaries ====
    "n_domains": 4,                     # n_domains - 1 wedges + 1 central core
    "core_frac": 0.35,                  # core radius as a fraction of sphere radius
    "core_bump_amp": 0.12,              # core-boundary noise amplitude (0 = perfect sphere)
    "wedge_angle_amp_deg": 12.0,        # wedge-boundary angular noise, degrees
    "noise_terms": 6,                   # sinusoidal terms summed for the boundary noise field
    "noise_freq_range": (0.8, 2.2),     # spatial-frequency range of that noise field
    "boundary_fuzz_width_deg": 0.0,     # wedge<->wedge relabeling band, degrees (0 = off)
    "boundary_fuzz_flip_prob": 0.0,     # relabel probability within that band
    "core_fuzz_width_um": 0.0,          # core<->wedge relabeling band, microns (0 = off)
    "core_fuzz_flip_prob": 0.0,         # relabel probability within that band

    # ==== Cell type composition by domain ====
    "n_cell_types": 4,                  # number of cell types (K)
    "domain_type_mix": None,            # (n_domains, n_cell_types) composition matrix pi_d; see warning below

    # ==== Gene programs and per-cell gene expression ====
    "marker_genes_per_type": 40,        # marker genes assigned to each cell type
    "noise_gene_frac": 0.10,            # fraction of the gene panel carrying no cell-type signal
    "shared_marker_frac": 0.25,         # fraction of each type's markers shared with other types
    "base_gene_lognormal": (0.7, 0.7),  # (mu, sigma) of each gene's baseline expression lambda_g
    "marker_foldchange": 3.5,           # multiplier for a cell type's own unique markers (large m_{t,g})
    "shared_marker_foldchange": 2.5,    # multiplier for markers shared across all cell types (intermediate m_{t,g})
    "noise_scale": 0.9,                 # multiplier for genes with no cell-type signal at all (weak m_{t,g})
    "cell_size_lognormal": (0.0, 0.35), # (mu, sigma) of each cell's size factor s_i
    "domain_size_factors": None,        # delta_di; None -> 1.0 for every domain (no effect)
    "theta": 25.0,                      # NB dispersion center theta_g (Var = mean + mean^2/theta)
    "theta_jitter": 2.0,                # per-gene jitter of theta, Normal(theta, theta_jitter)

    # ==== Transcript-level molecule simulation ====
    "inside_prob": 0.95,                # target fraction of a cell's molecules within its own radius r_i

    # ==== Cell-level observation and spillover ====
    "assign_k": 8,                      # nearest cells considered when reassigning transcripts by containment

    # ==== Platform-specific capture windows ====
    "xenium_capture_window_um": (12000.0, 24000.0),  # conceptual Xenium-like window (12x24 mm); not applied to cropping
    "capture_window_um": (300.0, 300.0),             # Visium/VisiumHD-like capture window, microns
    "capture_window_center_um": (0.0, 0.0),          # capture window center, microns
    "bin_size_um": 25.0,                             # VisiumHD-like bin width, microns
    "spot_spacing_um": 100.0,                        # Visium-like spot center-to-center spacing, microns
    "spot_radius_um": 27.5,                          # Visium-like spot capture radius, microns

    # ==== Slicing, batch effects, and unaligned coordinates ====
    "n_slices": 5,                      # number of slices along each requested axis
    "slice_axes": ("Z",),               # which axes to slice along ("X", "Y", "Z")
    "batch_sigma": 0.15,                # stdev of per-slice, per-gene log-fold-change epsilon_{s,g}
    "max_deg": 180.0,                   # per-slice rotation ~ Uniform[-max_deg, +max_deg]
    "max_shift": 200.0,                 # per-slice translation ~ Uniform[-max_shift, +max_shift], per axis
    "base_seed_unaligned": 12345,       # base seed for the per-slice unaligned-coordinate RNG

    # ==== Simulation outputs ====
    "output_modalities": ("cell", "bin", "spot"),  # which of {"cell", "bin", "spot"} to generate
    "sparse_X": True,                              # store count matrices as sparse arrays
    "seed": 2025,                                  # random seed
}

len(config)


## One-shot: pass the whole dictionary at once

`simulate_3d_molecule_sphere_multires(**config)` builds the shared 3D tissue
once, then slices/captures/applies batch effects for every requested
modality/axis combination in `config["output_modalities"]` /
`config["slice_axes"]`.


In [ ]:
sim = sim_app.simulate_3d_molecule_sphere_multires(**config)

print("top-level keys:", list(sim.keys()))
bin_adata = sim["bin_adatas"]["Z"]
bin_adata


## Inspect what was actually resolved

`sim["meta"]` records the fully-resolved simulation parameters -- useful for
confirming exactly what ran, the same way `sim_app.describe()` does for the
higher-level API.


In [ ]:
for key in ("sphere_R_um", "n_domains", "n_cell_types", "n_slices", "n_cells"):
    print(f"{key}: {sim['meta'][key]}")


## Two-step: reuse one simulated tissue across multiple sectioning configs

Splitting the same dictionary in two lets you simulate the tissue, cells,
domains, and gene panel **once**, then section it multiple different ways
without re-simulating any of that. This is the pattern used when you need,
say, several `n_slices`/`max_deg` combinations from the same underlying
tissue.

`simulate_3d_molecule_sphere_base(**kwargs)` is exactly
`simulate_3d_molecule_sphere_multires(**kwargs, _section=False)` -- it stops
right after tissue/cell/gene/molecule generation.


In [ ]:
# Step 1: everything up through molecule generation. capture_window_um stays IN
# tissue_config even though section_3d_molecule_sphere also takes it -- it's recorded
# in base["meta"] and inherited by any section_3d_molecule_sphere call that doesn't
# override it, so leaving it out here would silently fall back to that function's own
# much larger default (6500, 6500) instead of this tutorial's small (300, 300) window.
section_only_keys = {
    "n_slices", "batch_sigma",
    "bin_size_um", "spot_spacing_um", "spot_radius_um",
    "max_deg", "max_shift", "base_seed_unaligned",
    "slice_axes",
}
tissue_config = {k: v for k, v in config.items() if k not in section_only_keys}

base = sim_app.simulate_3d_molecule_sphere_base(**tissue_config)
print("base keys:", list(base.keys()))

# Step 2a: section along Z with the tutorial's usual settings
sim_z = sim_app.section_3d_molecule_sphere(
    base,
    n_slices=config["n_slices"],
    batch_sigma=config["batch_sigma"],
    capture_window_um=config["capture_window_um"],
    bin_size_um=config["bin_size_um"],
    max_deg=config["max_deg"],
    max_shift=config["max_shift"],
    output_modalities=config["output_modalities"],
    slice_axes=("Z",),
)

# Step 2b: reuse the SAME base tissue, section again with different settings -- no
# re-simulation of cells/domains/genes. bin_size_um is passed explicitly here because,
# unlike capture_window_um, it does NOT inherit from base if omitted -- it would fall
# back to section_3d_molecule_sphere's own default (8.0) instead.
sim_x_more_slices = sim_app.section_3d_molecule_sphere(
    base,
    n_slices=8,               # different from tissue_config -- no need to touch step 1 at all
    bin_size_um=config["bin_size_um"],
    max_deg=30.0,             # a much gentler misalignment for this pass
    output_modalities=config["output_modalities"],
    slice_axes=("X",),
)

print("Z sectioning, n_slices=5:", sim_z["bin_adatas"]["Z"].n_obs, "bins")
print("X sectioning, n_slices=8:", sim_x_more_slices["bin_adatas"]["X"].n_obs, "bins")


## Change anything: parameters `generate_data()` can't reach at all

Two examples of low-level-only parameters, both discussed in
`LOW_LEVEL_SIMULATOR.md`:

**1. `domain_type_mix` -- the uniform-fallback warning.** If you change
`n_domains`/`n_cell_types` away from `4`/`4` and don't supply
`domain_type_mix` yourself, every domain silently gets an identical, fully
uniform cell-type distribution -- no domain-specific enrichment at all. This
now raises a `UserWarning` so it's impossible to miss:


In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    _ = sim_app.simulate_3d_molecule_sphere_multires(
        sphere_R_um=150.0, n_cells=200, n_domains=3, n_cell_types=5,
        capture_window_um=(200.0, 200.0), output_modalities=("cell",), slice_axes=("Z",),
    )
    domain_mix_warnings = [str(w.message) for w in caught if "domain_type_mix" in str(w.message)]

print(domain_mix_warnings[0] if domain_mix_warnings else "(no warning raised)")


Fix it by supplying your own `(n_domains, n_cell_types)` matrix -- each row
gets renormalized to a probability distribution automatically, so it doesn't
need to sum to exactly 1:


In [ ]:
my_mix = np.array([
    [0.6, 0.2, 0.1, 0.05, 0.05],   # domain 0: enriched for type 0
    [0.1, 0.6, 0.2, 0.05, 0.05],   # domain 1: enriched for type 1
    [0.2, 0.2, 0.2, 0.2, 0.2],     # domain 2 (core): uniform, on purpose
])

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    sim_custom_mix = sim_app.simulate_3d_molecule_sphere_multires(
        sphere_R_um=150.0, n_cells=200, n_domains=3, n_cell_types=5,
        domain_type_mix=my_mix,
        capture_window_um=(200.0, 200.0), output_modalities=("cell",), slice_axes=("Z",),
    )
    print("warnings raised:", len(caught))


**2. `domain_size_factors` -- a domain-wide expression shift, independent of
cell type.** Defaults to `None` (every domain gets a factor of `1.0`, i.e. no
effect at all) -- none of the paper's own datasets ever turn this on. Compare
mean expression per domain with it off vs. on:


In [ ]:
def mean_expr_by_domain(adata):
    import pandas as pd
    totals = np.asarray(adata.X.sum(axis=1)).ravel()
    return pd.Series(totals, index=adata.obs["domain_true"].values).groupby(level=0).mean()

base_run = sim_app.simulate_3d_molecule_sphere_multires(
    sphere_R_um=150.0, n_cells=1000, n_domains=3, n_cell_types=3,
    domain_type_mix=np.full((3, 3), 1 / 3),   # flat composition, so any difference below is from domain_size_factors alone
    capture_window_um=(200.0, 200.0), output_modalities=("cell",), slice_axes=("Z",), seed=7,
)
boosted_run = sim_app.simulate_3d_molecule_sphere_multires(
    sphere_R_um=150.0, n_cells=1000, n_domains=3, n_cell_types=3,
    domain_type_mix=np.full((3, 3), 1 / 3),
    domain_size_factors=[1.0, 1.0, 1.6],      # domain 2 runs 60% higher, independent of cell type
    capture_window_um=(200.0, 200.0), output_modalities=("cell",), slice_axes=("Z",), seed=7,
)

print("domain_size_factors=None:")
print(mean_expr_by_domain(base_run["adata_cell_true"]))
print()
print("domain_size_factors=[1.0, 1.0, 1.6]:")
print(mean_expr_by_domain(boosted_run["adata_cell_true"]))


## Save outputs

Same `sim_app.save(...)` helper as the higher-level API -- it just writes an
`AnnData` to `.h5ad` and returns the path.


In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

output_path = sim_app.save(sim["bin_adatas"]["Z"], output_dir / "lowlevel_bin_z.h5ad")
output_path
